- 术语

- seq2seq(2014年)的问题：
    - 编码器的隐藏层 传递解码器：
        - 信息瓶颈: 源序列压缩为固定长度的问题（num_layers, seq_len, hidden_size）:信息丢失
        - 梯度消失：RNN,LSTM，GRU
        - 

Q（我在找什么？ query:中心词）
K (我提供什么？ key：与中心城相关的词)
V (实际的值)

- 记忆力$\to$注意力
    - 记忆力:历史序列的预测影响力
    - 注意力：重要的序列记忆才有价值

- 注意力的三种运算：
    - 点积
    - 加法(注意力)
    - 缩放点击（Transformer：自注意力）

# 点积注意力 

In [ ]:
import torch
def dot_product_attention(query, keys, values):
    # query: (hidden_size,)
    # keys: (src_len, hidden_size)
    # values: (src_len, hidden_size)
    
    # 计算得分: 点积
    scores = torch.matmul(keys, query)  # (src_len,)
    
    # Softmax归一化
    attention_weights = torch.softmax(scores, dim=0)
    
    # 加权求和
    context = torch.matmul(attention_weights, values)
    
    return context, attention_weights

# 加法注意力

# 注意力Seq2Sqe

## 1. 编码器

In [ ]:
import torch
from torch import nn
class GRUEncoder(nn.Module):
    def __init__(self, input_vocab_size, embedding_size, hidden_size, num_layers=2, dropout=0.5, bidirectional=True):
        super(GRUEncoder, self).__init__()
        self.hidden_size   = hidden_size
        self.num_layers    = num_layers
        self.bidirectional = bidirectional

        # 词嵌入
        self.embedding = nn.Embedding(input_vocab_size, embedding_size, padding_idx=0)
        self.dropout   = nn.Dropout(dropout)
        
        # 双向GRU
        self.gru       = nn.GRU(embedding_size, hidden_size, num_layers, 
                         batch_first=True, dropout=dropout, bidirectional=bidirectional)
        
        # 如果是双向，需要将隐藏状态维度减半
        if bidirectional:
            self.fc_hidden = nn.Linear(hidden_size * 2, hidden_size)    #双向 两层变一层·
        
    def forward(self, x):
        """
        Args:
            x: 输入序列 [batch_size, src_len]
        Returns:
            outputs: 编码器所有时刻的输出 [batch_size, src_len, hidden_size]
            hidden: 最终隐藏状态 [num_layers, batch_size, hidden_size]
        """
        embedded        = self.dropout(self.embedding(x))
        
        # GRU前向传播
        outputs, hidden = self.gru(embedded)
        
        # 如果使用双向GRU，需要处理隐藏状态
        if self.bidirectional:
            # outputs: [batch_size, src_len, hidden_size * 2]
            # 将双向输出合并
            outputs = self.fc_hidden(outputs)
            
            # hidden: [num_layers * 2, batch_size, hidden_size]
            # 合并双向的隐藏状态
            hidden = hidden.view(self.num_layers, 2, -1, self.hidden_size)
            hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=-1)
            hidden = self.fc_hidden(hidden)
        
        return outputs, hidden

In [7]:
import torch
# 模拟源序列
src = torch.randint(0, 100, (64, 32), dtype=torch.long)
# 模拟目标序列
# tgt = torch.randint(0, 100, (64, 36), dtype=torch.long)

# 构建编码器对象
input_dim = 30000   # 词袋大小
encoder_embedding_dim = 128   # 词嵌入大小
hidden_dim = 32   # 隐藏层大小
n_layers = 2
encoder = GRUEncoder(
    input_dim,
    encoder_embedding_dim,
    hidden_dim,
    n_layers,
)

outputs, hiddens = encoder(src)
print("outputs:", outputs.shape)
print("hiddens:", hiddens.shape)

outputs: torch.Size([64, 32, 32])
hiddens: torch.Size([2, 64, 32])


## 2. 注意力

In [8]:
class AdditiveAttention(nn.Module):
    """加法注意力机制 - 适配GRU"""
    def __init__(self, hidden_size):
        # hidden_size是解码器的隐藏层维度长度
        super(AdditiveAttention, self).__init__()
        self.hidden_size = hidden_size
        
        # 定义注意力层的参数
        self.Q = nn.Linear(hidden_size, hidden_size, bias=False)
        self.K = nn.Linear(hidden_size, hidden_size, bias=False)
        self.V = nn.Linear(hidden_size, 1,bias=False)
        
    def forward(self, decoder_hidden, encoder_outputs):
        """
        Args:
            decoder_hidden: 解码器隐藏状态 [batch_size, hidden_size]  也是编码器的输出的隐藏状态
            encoder_outputs: 编码器所有时刻的输出 [batch_size, src_len, hidden_size]
        Returns:
            context: 上下文向量 [batch_size, hidden_size]
            attention_weights: 注意力权重 [batch_size, src_len]
        """
        batch_size = encoder_outputs.shape[0]
        src_len = encoder_outputs.shape[1]
        
        # 扩展解码器隐藏状态以匹配编码器输出的序列长度
        decoder_hidden = decoder_hidden.unsqueeze(1).expand(-1, src_len, -1)
        
        # 计算能量分数
        energy = torch.tanh(self.Q(decoder_hidden) + self.K(encoder_outputs))
        scores = self.V(energy).squeeze(-1)
        
        # 计算注意力权重
        attention_weights = torch.softmax(scores, dim=1)
        
        # 计算上下文向量
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, attention_weights

In [9]:
hidden_size = 32   # 解码器隐藏层与编码器保持一致。
attn = AdditiveAttention(hidden_size)
context, attention_weights = attn(hiddens[-1, :, :], outputs)  # 使用最后一个hiddens作为解码器的隐藏层
print(context.shape)

torch.Size([64, 32])


## 3. 解码器（使用注意力）